# 03 — Multi-Head Attention
### Starter Notebook

**Learning objectives**
- Understand why multiple heads are better than one
- Implement `MultiHeadAttention` from scratch
- Explore Grouped-Query Attention (GQA) — the trick used in Gemma/LLaMA
- Visualise what different heads specialise in

---

In [ ]:
import sys, os, math
sys.path.insert(0, os.path.abspath('../..'))

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional, Tuple

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Part A — Motivation: One Head vs Many

A single attention head produces one weighted average of values. Multiple heads run in **parallel** on smaller subspaces, letting the model attend to different things simultaneously:
- Head 1 might capture syntactic dependencies (subject → verb)
- Head 2 might track coreference ("it" → "the cat")
- Head 3 might handle positional patterns

$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W^O$$
$$\text{head}_i = \text{Attention}(Q W_i^Q,\; K W_i^K,\; V W_i^V)$$

## Part B — Implement MultiHeadAttention

### Exercise B1
Complete the class below. The key steps are:
1. Project Q, K, V with separate weight matrices
2. Split the model dimension across heads
3. Run attention in parallel across all heads
4. Concatenate and project back

In [ ]:
from src.models.attention import scaled_dot_product_attention


class MyMultiHeadAttention(nn.Module):
    """
    Multi-Head Attention from scratch.

    Args:
        d_model  : total model dimension (e.g. 512)
        n_heads  : number of heads (e.g. 8)
        dropout  : attention dropout probability
    """

    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.dropout = dropout

        # TODO: define w_q, w_k, w_v, w_o  (all nn.Linear(d_model, d_model))

    def forward(
        self,
        query: torch.Tensor,          # (batch, seq_q, d_model)
        key:   torch.Tensor,          # (batch, seq_k, d_model)
        value: torch.Tensor,          # (batch, seq_k, d_model)
        mask:  Optional[torch.Tensor] = None,
        return_attention: bool = False,
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        batch = query.size(0)

        # TODO 1: project Q, K, V and reshape to (batch, n_heads, seq, d_k)
        q = None
        k = None
        v = None

        # TODO 2: run scaled_dot_product_attention on all heads at once
        output, attn_weights = None, None

        # TODO 3: reshape output back to (batch, seq, d_model) and apply w_o
        output = None

        if return_attention:
            return output, attn_weights
        return output, None


# Quick shape test
mha = MyMultiHeadAttention(d_model=128, n_heads=4)
x = torch.randn(2, 10, 128)   # batch=2, seq=10
out, w = mha(x, x, x, return_attention=True)
if out is not None:
    print(f'Output : {out.shape}    # expect (2, 10, 128)')
    print(f'Weights: {w.shape}   # expect (2, 4, 10, 10)')
else:
    print('Not yet implemented.')

### Exercise B2 — Parameter Count

How many parameters does an MHA layer with `d_model=512, n_heads=8` have?  
Calculate by hand first, then verify with `sum(p.numel() for p in mha.parameters())`.

In [ ]:
# Hand calculation:
# Each of W_q, W_k, W_v, W_o is a (d_model × d_model) matrix with bias
d_model = 512
expected_params = None  # TODO: calculate

mha_512 = MyMultiHeadAttention(d_model=512, n_heads=8)
actual_params = sum(p.numel() for p in mha_512.parameters())
print(f'Expected : {expected_params}')
print(f'Actual   : {actual_params}')

## Part C — Visualise Head Specialisation

In [ ]:
from src.models.attention import MultiHeadAttention

sentence = "the quick brown fox jumps over the lazy dog"
words = sentence.split()
vocab = {w: i for i, w in enumerate(sorted(set(words)))}

d_model, n_heads = 64, 4
embed = nn.Embedding(len(vocab), d_model)
tokens = torch.tensor([vocab[w] for w in words])
x = embed(tokens).unsqueeze(0)   # (1, seq, d_model)

mha = MultiHeadAttention(d_model=d_model, n_heads=n_heads)
_, weights = mha(x, x, x, return_attention=True)
# weights: (1, n_heads, seq, seq)

fig, axes = plt.subplots(1, n_heads, figsize=(4 * n_heads, 4))
for h, ax in enumerate(axes):
    w_np = weights[0, h].detach().numpy()
    im = ax.imshow(w_np, cmap='Blues', vmin=0, vmax=w_np.max())
    ax.set_title(f'Head {h+1}')
    ax.set_xticks(range(len(words))); ax.set_xticklabels(words, rotation=45, ha='right', fontsize=7)
    ax.set_yticks(range(len(words))); ax.set_yticklabels(words, fontsize=7)
    plt.colorbar(im, ax=ax)

plt.suptitle(f'Multi-Head Attention ({n_heads} heads)', fontsize=12)
plt.tight_layout()
plt.show()

## Part D — Grouped-Query Attention (GQA)

Modern LLMs (Gemma, LLaMA-3, Mistral) use **fewer KV heads than Q heads** to reduce memory during inference.  
Each KV head is shared by `n_heads // n_kv_heads` query heads.

### Exercise D1
Compute the KV cache memory savings for a typical LLM configuration.

In [ ]:
from src.models.attention import GroupedQueryAttention

# Test GQA
gqa = GroupedQueryAttention(d_model=128, n_heads=8, n_kv_heads=2)
x = torch.randn(2, 10, 128)
out, _ = gqa(x, x, x)
print(f'GQA output: {out.shape}   # expect (2, 10, 128)')

# Memory analysis
def kv_cache_size_mb(
    n_layers: int, seq_len: int, n_kv_heads: int, d_k: int,
    bytes_per_elem: int = 2,  # float16
) -> float:
    """KV cache size in MB."""
    # K cache + V cache, each: (n_layers, seq_len, n_kv_heads, d_k)
    elements = 2 * n_layers * seq_len * n_kv_heads * d_k
    return elements * bytes_per_elem / 1e6


# Gemma-7B-like config
n_layers, seq_len, d_model, n_heads = 28, 2048, 3072, 16
d_k = d_model // n_heads

mha_mb  = kv_cache_size_mb(n_layers, seq_len, n_heads,   d_k)   # MHA
gqa2_mb = kv_cache_size_mb(n_layers, seq_len, 2,         d_k)   # GQA with 2 KV heads
gqa1_mb = kv_cache_size_mb(n_layers, seq_len, 1,         d_k)   # MQA  with 1 KV head

print(f'\nKV-cache memory at seq_len={seq_len}:')
print(f'  MHA  ({n_heads:2d} KV heads): {mha_mb:7.1f} MB')
print(f'  GQA  ( 2 KV heads): {gqa2_mb:7.1f} MB   ({mha_mb/gqa2_mb:.1f}x smaller)')
print(f'  MQA  ( 1 KV head ): {gqa1_mb:7.1f} MB   ({mha_mb/gqa1_mb:.1f}x smaller)')

## Summary

| Variant | KV heads | Quality | KV cache |
|---------|----------|---------|----------|
| MHA | = Q heads | Best | Largest |
| GQA | 2–8 | Near-MHA | Moderate |
| MQA | 1 | Slight drop | Smallest |

**Next:** `04_positional_encoding_starter.ipynb` — how do transformers know token order?